In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from sklearn.model_selection import train_test_split

In [2]:
train = pd.read_csv('train.csv')
songs = pd.read_csv('songs.csv')
save = pd.read_csv('save_for_later.csv')
test = pd.read_csv('test.csv')

In [3]:
train_merged = train.merge(songs, on='song_id', how='left')
test_merged  = test.merge(songs, on='song_id', how='left')

In [4]:
save['saved_flag'] = 1
train_merged = train_merged.merge(save[['customer_id','song_id','saved_flag']],
                                  on=['customer_id','song_id'], how='left')
train_merged['saved_flag'].fillna(0, inplace=True)

test_merged = test_merged.merge(save[['customer_id','song_id','saved_flag']],
                                on=['customer_id','song_id'], how='left')
test_merged['saved_flag'].fillna(0, inplace=True)

C:\Users\Tanushree\AppData\Local\Temp\ipykernel_2416\2580303461.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_merged['saved_flag'].fillna(0, inplace=True)
C:\Users\Tanushree\AppData\Local\Temp\ipykernel_2416\2580303461.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.


In [5]:
missing_rows = train_merged[train_merged.isnull().any(axis=1)]
print(missing_rows)

       customer_id  song_id  score platform_id  released_year language  \
4            K4115     8452      5       V5946         1987.0      NaN   
16          H45532     8793      1     W342667         2005.0      NaN   
30          H19382     1621      4    V6452796         2008.0      NaN   
75          G27611     5845      4      W56157         2003.0      NaN   
106         M28387      132      5       Q3431         2003.0      NaN   
...            ...      ...    ...         ...            ...      ...   
709986       H2682     6457      5    W1055617         1969.0      NaN   
710022      K50565     2256      3     Q381421         2004.0      NaN   
710034      I44583     1826      5      W98427         1992.0      NaN   
710071      G25241     1054      5   W13125947         2012.0      NaN   
710107      G41961     2397      4      W76237         1985.0      NaN   

        number_of_comments  saved_flag  
4                    762.0         0.0  
16                   435.0   

In [6]:
for df_name, df in [('train', train), ('test', test), ('songs', songs), ('save', save)]:
    print(f"Missing values in {df_name}:")
    print(df.isnull().sum())
    print()

Missing values in train:
customer_id    0
song_id        0
score          0
dtype: int64

Missing values in test:
customer_id    0
song_id        0
dtype: int64

Missing values in songs:
song_id                  0
platform_id              0
released_year           16
language              1076
number_of_comments       0
dtype: int64

Missing values in save:
customer_id    0
song_id        0
saved_flag     0
dtype: int64



In [7]:
train_merged['released_year'].fillna(-1, inplace=True)
test_merged['released_year'].fillna(-1, inplace=True)

C:\Users\Tanushree\AppData\Local\Temp\ipykernel_2416\471656590.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_merged['released_year'].fillna(-1, inplace=True)
C:\Users\Tanushree\AppData\Local\Temp\ipykernel_2416\471656590.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy

In [8]:
train_merged['language'].fillna('unknown', inplace=True)
test_merged['language'].fillna('unknown', inplace=True)

C:\Users\Tanushree\AppData\Local\Temp\ipykernel_2416\3132875727.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_merged['language'].fillna('unknown', inplace=True)
C:\Users\Tanushree\AppData\Local\Temp\ipykernel_2416\3132875727.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a 

In [9]:
print(train_merged.head())

  customer_id  song_id  score platform_id  released_year language  \
0      O29219     3459      3      P49540         1782.0      eng   
1      I50343     5326      4     W216377         1981.0      eng   
2      N42888      236      5       X1898         1997.0    en-US   
3       F5740      724      4    W2233407         2008.0      eng   
4       K4115     8452      5       V5946         1987.0  unknown   

   number_of_comments  saved_flag  
0              1066.0         0.0  
1              1119.0         0.0  
2             10439.0         0.0  
3              3500.0         0.0  
4               762.0         0.0  


In [10]:
current_year = 2025
train_merged['song_age'] = current_year - train_merged['released_year']
test_merged['song_age']  = current_year - test_merged['released_year']

In [11]:
all_languages = pd.concat([train_merged['language'], test_merged['language']]).unique()
language_map = {lang: idx+1 for idx, lang in enumerate(all_languages)}
train_merged['language'] = train_merged['language'].map(language_map)
test_merged['language']  = test_merged['language'].map(language_map)

In [12]:
print(train_merged.head())

  customer_id  song_id  score platform_id  released_year  language  \
0      O29219     3459      3      P49540         1782.0         1   
1      I50343     5326      4     W216377         1981.0         1   
2      N42888      236      5       X1898         1997.0         2   
3       F5740      724      4    W2233407         2008.0         1   
4       K4115     8452      5       V5946         1987.0         3   

   number_of_comments  saved_flag  song_age  
0              1066.0         0.0     243.0  
1              1119.0         0.0      44.0  
2             10439.0         0.0      28.0  
3              3500.0         0.0      17.0  
4               762.0         0.0      38.0  


In [13]:
numeric_cols = ['released_year', 'song_age', 'number_of_comments', 'saved_flag']
for col in numeric_cols:
    Q1 = train_merged[col].quantile(0.25)
    Q3 = train_merged[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    train_merged[col] = train_merged[col].clip(lower, upper)
    test_merged[col]  = test_merged[col].clip(lower, upper)

In [14]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
train_merged[numeric_cols] = scaler.fit_transform(train_merged[numeric_cols])
test_merged[numeric_cols]  = scaler.transform(test_merged[numeric_cols])

In [15]:
from sklearn.preprocessing import LabelEncoder

customer_enc = LabelEncoder()
song_enc     = LabelEncoder()

train_merged['customer_enc'] = customer_enc.fit_transform(train_merged['customer_id'])
train_merged['song_enc']     = song_enc.fit_transform(train_merged['song_id'])

test_merged['customer_enc'] = customer_enc.transform(test_merged['customer_id'])
test_merged['song_enc']     = song_enc.transform(test_merged['song_id'])


In [16]:
print(train_merged.head())

  customer_id  song_id  score platform_id  released_year  language  \
0      O29219     3459      3      P49540       0.000000         1   
1      I50343     5326      4     W216377       0.578947         1   
2      N42888      236      5       X1898       0.766082         2   
3       F5740      724      4    W2233407       0.894737         1   
4       K4115     8452      5       V5946       0.649123         3   

   number_of_comments  saved_flag  song_age  customer_enc  song_enc  
0            0.030927         0.0  1.000000         13168      3458  
1            0.032469         0.0  0.421053          5393      5325  
2            0.303628         0.0  0.233918         12189       235  
3            0.101743         0.0  0.105263          1306       723  
4            0.022083         0.0  0.350877          7967      8451  


In [18]:
train_merged.to_csv("cleaned_dataset.csv", index=False)
test_merged.to_csv("cleaned_testing.csv",index=False)

In [17]:
feature_cols = ['customer_enc', 'song_enc', 'released_year', 'song_age',
                'language', 'number_of_comments', 'saved_flag']

X = train_merged[feature_cols]
y = train_merged['score']
X_test = test_merged[feature_cols]

In [22]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)